# LegalIR Task 1: High-Recall Vietnamese Legal Information Retrieval
## UIT Data Science Challenge 2026 — Kaggle Final Production Runner
**Pinned Runtime Commit:** `17829909630adcb3d46de1e0bc27e9a903d68749`

### Production Pipeline:
1. **Verification**: Exact runtime SHA & canonical dataset v2 verification.
2. **Immutable Bundle**: Cryptographic & semantic verification of production bundle.
3. **Training**: Train one final `BAAI/bge-reranker-v2-m3` LoRA adapter on all 7,000 queries.
4. **Public Rerank & Fusion**: Rerank public candidates under frozen fusion winner.
5. **Submission**: Strict validation ($1 \le |answer| \le 5$, unique official doc IDs) and `submission.zip` packaging.

### Invariants:
- Learned Parameter Budget: < 4,000,000,000 (4B)
- Zero PyTorch reinstallation
- Exactly 1,000 public test queries covered


In [ ]:
# ==============================================================================
# Cell 1: Hardware & Environment Preflight
# ==============================================================================
import os
import sys
import subprocess
import torch

print(f"[+] Python Version : {sys.version.split()[0]}")
print(f"[+] PyTorch Version: {torch.__version__}")
print(f"[+] CUDA Available : {torch.cuda.is_available()}")
RUN_MODE = os.environ.get("LEGALIR_RUN_MODE", "full")  # 'full' or 'smoke'
print(f"[*] Execution Run Mode: {RUN_MODE}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        vram_gb = prop.total_memory / (1024**3)
        print(f"    - GPU {i}: {prop.name} | Total VRAM: {vram_gb:.2f} GB")
else:
    print("[!] Running on CPU (testing/smoke mode).")


In [ ]:
# ==============================================================================
# Cell 2: Repository Bootstrap & Minimal Dependencies (Zero Torch Reinstall)
# ==============================================================================
from pathlib import Path

CWD = Path.cwd()
possible_repo_paths = [
    CWD,
    CWD / "LegalIR",
    Path("/kaggle/working/LegalIR"),
    Path("/kaggle/working"),
]

# Pinned runtime: git checkout 17829909630adcb3d46de1e0bc27e9a903d68749
EXPECTED_COMMIT = os.environ.get("LEGALIR_COMMIT_SHA", "17829909630adcb3d46de1e0bc27e9a903d68749")
REPO_ROOT = None
for p in possible_repo_paths:
    if (p / "scripts" / "run_kaggle_final.py").exists():
        REPO_ROOT = p.resolve()
        break

if REPO_ROOT is None:
    print("[*] Cloning LegalIR repository into /kaggle/working/LegalIR...")
    target_dir = Path("/kaggle/working/LegalIR")
    if not target_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/silent9669/LegalIR.git", str(target_dir)], check=True)
    if EXPECTED_COMMIT and EXPECTED_COMMIT != "main":
        subprocess.run(["git", "fetch", "--all"], cwd=target_dir, check=True)
        subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=target_dir, check=True)
    REPO_ROOT = target_dir.resolve()

actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT).decode("utf-8").strip()
if EXPECTED_COMMIT and EXPECTED_COMMIT != "main" and actual_commit != EXPECTED_COMMIT:
    subprocess.run(["git", "fetch", "--all"], cwd=REPO_ROOT, check=True)
    subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_ROOT, check=True)
    actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT).decode("utf-8").strip()
    if actual_commit != EXPECTED_COMMIT:
        raise RuntimeError(f"Fail-closed commit pin violation: expected {EXPECTED_COMMIT}, got {actual_commit}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Install minimal dependencies without touching PyTorch
required_pkgs = []
for mod, pkg in [
    ("lightgbm", "lightgbm"),
    ("sentencepiece", "sentencepiece"),
    ("bm25s", "bm25s"),
    ("pyvi", "pyvi"),
    ("peft", "peft"),
    ("accelerate", "accelerate"),
    ("faiss", "faiss-cpu"),
    ("psutil", "psutil"),
]:
    try:
        __import__(mod)
    except ImportError:
        required_pkgs.append(pkg)

if required_pkgs:
    print(f"[*] Installing missing packages: {required_pkgs}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location"] + required_pkgs, check=True)
print(f"[+] Repositories and dependencies ready on commit {actual_commit}.")


In [ ]:
# ==============================================================================
# Cell 3: Canonical Dataset Discovery & Preflight Metadata Check (pq.ParquetFile)
# ==============================================================================
import json
import pyarrow.parquet as pq
from src.pipeline.kaggle_train import discover_data_dir, discover_public_test_file

possible_datasets = [
    Path("/kaggle/input/task1-canonical-v2"),
    Path("/kaggle/input/legalir-task1-clean-data"),
    REPO_ROOT / "data/task1_canonical_v2",
    Path("data/task1_canonical_v2"),
]
data_dir = next((p for p in possible_datasets if (p / "documents.parquet").exists() or (p / "public-official.json").exists()), possible_datasets[0])
public_file = discover_public_test_file(repo_root=REPO_ROOT)

docs_rows = pq.ParquetFile(data_dir / "documents.parquet").metadata.num_rows if (data_dir / "documents.parquet").exists() else 0
public_data = json.loads(public_file.read_text(encoding="utf-8")) if (public_file and public_file.exists()) else {}
public_rows = len(public_data)
print(f"[+] Canonical Data Directory: {data_dir}")
print(f"[+] Documents Count         : {docs_rows:,} (expected 8,532)")
print(f"[+] Public Queries Count    : {public_rows:,} (expected 1,000)")
if public_rows != 1000 and RUN_MODE == "full":
    raise ValueError(f"Dataset identity mismatch: public queries count is {public_rows}, expected 1,000")

possible_bundles = [
    Path("/kaggle/input/legalir-production-bundle"),
    Path("/kaggle/input/production-bundle"),
    REPO_ROOT / "artifacts/bundle/production",
    Path("artifacts/bundle/production"),
]
bundle_dir = next((p for p in possible_bundles if (p / "production_lock.json").exists()), possible_bundles[0])

output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else REPO_ROOT / "artifacts/submission"

cmd = [
    sys.executable,
    str(REPO_ROOT / "scripts/run_kaggle_final.py"),
    "--dataset-dir", str(data_dir),
    "--bundle-dir", str(bundle_dir),
    "--output-dir", str(output_dir),
]
if RUN_MODE in ("smoke", "gpu_smoke", "mock"):
    cmd.append("--mock")
print(f"[*] Executing Kaggle Final: {' '.join(cmd)}")
subprocess.run(cmd, check=True)


In [ ]:
# ==============================================================================
# Cell 4: Verify Final Submission Artifact
# ==============================================================================
zip_candidates = [
    Path("/kaggle/working/submission.zip"),
    output_dir / "submission.zip",
]
zip_path = next((p for p in zip_candidates if p.is_file()), None)
if zip_path is None or not zip_path.is_file():
    raise FileNotFoundError("submission.zip was not found!")

print(f"[+] SUCCESS: Valid submission found at {zip_path} ({zip_path.stat().st_size:,} bytes).")
print("[+] Ready for official UIT competition submission.")
